# Spec-Craft with OpenCode: Interactive Agent & MCP Development

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akuroiwa/spec-craft/blob/main/docs/source/notebooks/spec_craft_opencode.ipynb)

Welcome to the **Spec-Craft & OpenCode** interactive tutorial on Google Colab!

This notebook provides a complete environment for **interactive Spec-Driven Development (SDD)** with **OpenCode** (an autonomous AI coding agent with native **Model Context Protocol (MCP)** support), **Spec-Craft** (Obsidian SDD MCP server), and **spec-kit** (`specify-cli`).

### 3-Tier SDD Architecture

```mermaid
flowchart TB
    OC["OpenCode<br/>Interactive AI Agent"]
    MCP["Model Context Protocol (MCP)"]
    SC["spec-craft<br/>FastMCP Server"]
    SK["spec-kit<br/>SDD Skills & Templates"]
    OBS["Obsidian<br/>Strategy Notes"]
    OLL["Ollama<br/>Gemma 4 QAT"]

    OC --> MCP
    MCP --> SC
    SC --> OBS
    OC --> SK
    OC --> OLL
```

### Interactive SDD Lifecycle Sequence

```mermaid
sequenceDiagram
    participant U as User (Developer)
    participant O as OpenCode TUI
    participant M as spec-craft MCP
    participant S as spec-kit
    participant F as File System

    U->>O: 1. Development Request (Task)
    O->>M: 2. Inspect Strategy (StoryboardParser)
    M->>F: Read Obsidian Strategy Note
    F-->>M: Strategy Content & Frontmatter
    M-->>O: Parsed Requirements & Sections
    O->>S: 3. Execute Tactical SDD Step
    S->>F: Generate Code & Test Suites
    O->>F: 4. Verify Correctness (pytest)
```

In this notebook, you will learn how to:
1. Set up **OpenCode**, **Ollama** (Gemma 4 QAT), **specify-cli**, and **spec-craft** in Colab.
2. Configure `opencode.json` with the `spec-craft` MCP server using the official OpenCode v2 configuration.
3. Launch an interactive web terminal using `colab-xterm` with **automatic session logging via `script`** for debugging and YouTube demo recording.
4. Follow the **step-by-step task breakdown protocol** to avoid monolithic timeout issues.
5. Download session execution logs directly from Google Colab.

In [ ]:
# ==============================================================================
# 1. Environment Setup (OpenCode, Ollama, specify-cli, colab-xterm, and spec-craft)
# ==============================================================================
import os, sys, time, subprocess, json

print("📦 1. Installing system dependencies & tools...")
!apt-get update -qq && apt-get install -y -qq build-essential curl zstd git pkg-config libssl-dev > /dev/null

print("📥 2. Installing OpenCode CLI & colab-xterm...")
!npm install -g opencode-ai
!pip install -q uv pytest colab-xterm

print("🚀 3. Installing and launching Ollama background service...")
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama in background
subprocess.Popen(["ollama", "serve"])

# Health check
for _ in range(12):
    time.sleep(1)
    try:
        res = subprocess.run(["curl", "-s", "http://127.0.0.1:11434"], capture_output=True, text=True)
        if "Ollama is running" in res.stdout:
            print("   ✅ Ollama server is active!")
            break
    except Exception:
        pass

print("🧠 4. Pulling Ollama model (Gemma 4 QAT)...")
# Model Choices:
# [Recommended Default for T4 GPU] 12B QAT (~7.2 GB VRAM):
!ollama pull gemma4:12b-it-qat
# --- Lightweight alternatives for Colab Free Tier / CPU ---
# [Fast / Low VRAM] 4B QAT (~3.8 GB VRAM):
# !ollama pull gemma4:e4b-it-qat
# [Ultra-Lightweight] 2B QAT (~2.2 GB VRAM):
# !ollama pull gemma4:e2b-it-qat

print("🛠️ 5. Installing specify-cli (spec-kit) & spec-craft...")
!uv tool install specify-cli || uv tool install specify-cli --force --from git+https://github.com/github/spec-kit.git
!uv pip install --system git+https://github.com/akuroiwa/spec-craft.git

# Set global PATH for current Python environment
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

print("🎉 Core environment ready!")

## 2. Configure OpenCode MCP Settings (`opencode.json`)

OpenCode supports local MCP servers via the `mcp` field in `~/.config/opencode/opencode.json` (see [OpenCode Configuration Docs](https://opencode.ai/v2/docs/config/) and [MCP Servers Docs](https://opencode.ai/v2/docs/mcp-servers/)).

In OpenCode v2:
- The local MCP server `command` is an array of strings: `["uv", "run", "spec-craft", "serve"]`.
- The local Ollama provider uses `@ai-sdk/openai-compatible` pointing to `http://127.0.0.1:11434/v1`.

In [ ]:
# ==============================================================================
# Dynamic OpenCode Configuration (Ollama Provider + spec-craft MCP Server)
# ==============================================================================
config_dir = os.path.expanduser("~/.config/opencode")
os.makedirs(config_dir, exist_ok=True)

opencode_config = {
  "$schema": "https://opencode.ai/config.json",
  "provider": {
    "ollama": {
      "npm": "@ai-sdk/openai-compatible",
      "name": "Ollama (local)",
      "options": {
        "baseURL": "http://127.0.0.1:11434/v1"
      },
      "models": {
        "gemma4:12b-it-qat": {
          "name": "Gemma 4 12B QAT"
        }
      }
    }
  },
  "model": "ollama/gemma4:12b-it-qat",
  "mcp": {
    "spec-craft": {
      "type": "local",
      "command": [
        "uv",
        "run",
        "spec-craft",
        "serve"
      ]
    }
  }
}

config_path = os.path.join(config_dir, "opencode.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(opencode_config, f, indent=2)

# Also place a copy in project root for local execution
with open("opencode.json", "w", encoding="utf-8") as f:
    json.dump(opencode_config, f, indent=2)

print(f"✅ OpenCode MCP configuration written to {config_path} and ./opencode.json")

## 3. Verify Prerequisites

Check that all environment tools and MCP bridges are ready using `spec-craft check-env`.

In [ ]:
!spec-craft check-env

## 4. Step-by-Step OpenCode Task Protocol

> **Important (Avoiding Timeout)**:
> Submitting a single monolithic prompt ("Draft strategy, parse MCP, implement module, write pytest, and run it all") can cause timeout on Colab's compute.
> Instead, execute tasks **step-by-step** in OpenCode:

| Step | Phase | OpenCode Prompt to Enter |
| :--- | :--- | :--- |
| **Task 1** | **MCP Verification** | `"Verify that the spec-craft MCP server is active. List available MCP tools without modifying files."` |
| **Task 2** | **Strategy Drafting** | `"Create an Obsidian strategy roadmap note in obsidian/strategy_task_tracker.md with frontmatter for a Markdown task tracker CLI."` |
| **Task 3** | **Strategy Inspection** | `"Use the spec-craft MCP StoryboardParser tool to inspect obsidian/strategy_task_tracker.md and summarize requirements."` |
| **Task 4** | **Implementation** | `"Implement task_tracker.py with TaskTracker class and analyze_tasks() function based on the strategy note."` |
| **Task 5** | **Testing & Verification** | `"Write unit tests in test_task_tracker.py covering empty, pending, completed, and mixed tasks, then run pytest."` |

## 5. Interactive Web Terminal with Session Recording (`colab-xterm`)

Execute the cell below to open an interactive Linux terminal within Google Colab.

### Terminal Execution Steps:
```bash
# 1. Initialize spec-kit in current directory
specify init --here --force --integration opencode

# 2. Launch OpenCode inside 'script' to record full terminal session logs
script -q -c 'opencode' /content/opencode-session.log
```

Inside OpenCode TUI, enter the step-by-step prompts from Section 4 above.

In [ ]:
# Load colabxterm extension and launch interactive terminal
%load_ext colabxterm
%xterm height=550

## 6. Verification & Test Execution

After completing tasks in OpenCode, verify that all unit tests pass directly in the notebook.

In [ ]:
# Execute unit tests
!pytest test_task_tracker.py -v

## 7. Inspect & Download OpenCode Session Logs

Review the recorded terminal log file (`opencode-session.log`) and download it to your local machine for analysis and reporting.

In [ ]:
# List and inspect generated log files
from pathlib import Path

log_files = list(Path("/content").glob("*.log"))
print(f"📁 Found {len(log_files)} log file(s):")
for log_path in log_files:
    print(f"   - {log_path.name}: {log_path.stat().st_size:,} bytes")

# Uncomment to download log file to your local computer:
# from google.colab import files
# if Path("/content/opencode-session.log").exists():
#     files.download("/content/opencode-session.log")

## 8. YouTube Video Demo & Recording Guide

For creating YouTube demonstration videos of Spec-Craft & OpenCode, here are recommended approaches:

### A. Screen Recording Tools
- **Screenity / Loom (Chrome Extension)**: Capture clean, high-resolution browser tabs with mouse click ripple effects and audio commentary.
- **OBS Studio**: Capture window or full screen with custom overlays, zoom effects, and high bitrate recording.

### B. Programmatic Automated Recording (Playwright)
You can also automate browser execution and record video using Playwright:
```python
# Example: Automated recording with Playwright
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)
    context = browser.new_context(record_video_dir="videos/", record_video_size={"width": 1280, "height": 720})
    page = context.new_page()
    page.goto("https://colab.research.google.com/github/akuroiwa/spec-craft/blob/main/docs/source/notebooks/spec_craft_opencode.ipynb")
    # Automate actions...
    context.close()
```

## Summary & Next Steps

Congratulations! You have experienced full interactive Spec-Driven Development with Spec-Craft & OpenCode:
- **Strategy (Obsidian)**: Captured user stories and technical requirements.
- **MCP Server (spec-craft)**: Exposed tools (`StoryboardParser`, `check-env`) directly to OpenCode.
- **Interactive TUI (OpenCode)**: Collaborated with the AI agent in real time with step-by-step execution.
- **Execution Logs**: Recorded complete session logs via `script` for review and verification.
- **Tactics (pytest)**: Verified correctness through automated test suites.

### Resources
- **GitHub Repository**: [https://github.com/akuroiwa/spec-craft](https://github.com/akuroiwa/spec-craft)
- **OpenCode Documentation**: [https://opencode.ai/v2/docs/mcp-servers/](https://opencode.ai/v2/docs/mcp-servers/)
- **Documentation**: [https://spec-craft.readthedocs.io/](https://spec-craft.readthedocs.io/)
- **Zenn Book / Articles**: [spec-craft Development Notes](https://zenn.dev/akuroiwa/books/dev-note-spec-craft)